#Natural Language Processing: Sentiment Analysis with Transformers

#Question 1: Sentiment Analysis with Transformers (IMDB)

##1. Install & Import

In [2]:
!pip install transformers datasets

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments



##2. Load Dataset

In [3]:
dataset = load_dataset("imdb")

train_data = dataset['train'].shuffle(seed=42).select(range(1000))
test_data = dataset['test'].shuffle(seed=42).select(range(500))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

##3. Load Tokenizer (DISTILBERT)

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length')

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

##4. Load Transformer Model

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


##5. Define Training Arguments

In [6]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1
)

In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

##6. Initialize Trainer

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

##7. Train the Model

In [9]:
trainer.train()
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.332122,0.870000,0.870775,0.852140,0.890244


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.3321220576763153,
 'eval_accuracy': 0.87,
 'eval_f1': 0.8707753479125249,
 'eval_precision': 0.8521400778210116,
 'eval_recall': 0.8902439024390244,
 'eval_runtime': 399.8912,
 'eval_samples_per_second': 1.25,
 'eval_steps_per_second': 0.313,
 'epoch': 1.0}

##8. Evaluate the model

In [10]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.3321220576763153, 'eval_accuracy': 0.87, 'eval_f1': 0.8707753479125249, 'eval_precision': 0.8521400778210116, 'eval_recall': 0.8902439024390244, 'eval_runtime': 399.303, 'eval_samples_per_second': 1.252, 'eval_steps_per_second': 0.313, 'epoch': 1.0}


##9. Generate Predictions

In [14]:
small_test = test_data.select(range(20))

predictions = trainer.predict(small_test)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


##10. Classification Report

In [12]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.85      0.87       254
           1       0.85      0.89      0.87       246

    accuracy                           0.87       500
   macro avg       0.87      0.87      0.87       500
weighted avg       0.87      0.87      0.87       500



##11. Sample Predictions

In [15]:
for i in range(5):
    print("Review:", small_test[i]['text'][:100])
    print("Actual:", y_true[i])
    print("Predicted:", y_pred[i])
    print("-"*50)


Review: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining Kin
Actual: 1
Predicted: 1
--------------------------------------------------
Review: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French a
Actual: 1
Predicted: 1
--------------------------------------------------
Review: This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good
Actual: 0
Predicted: 0
--------------------------------------------------
Review: I was truly and wonderfully surprised at "O' Brother, Where Art Thou?" The video store was out of al
Actual: 1
Predicted: 1
--------------------------------------------------
Review: This movie spends most of its time preaching that it is the script that makes the movie, but apparen
Actual: 0
Predicted: 0
--------------------------------------------------


######The Transformer model achieved an accuracy of 87% on the test dataset, indicating strong performance in sentiment classification. The precision, recall, and F1-score for both classes are well-balanced, demonstrating that the model effectively distinguishes between positive and negative reviews.

######The model shows slightly higher precision for negative reviews and slightly higher recall for positive reviews, but overall performance remains consistent across both classes. The balanced macro and weighted averages confirm that the model does not suffer from class imbalance issues.


#####The sample predictions demonstrate that the Transformer model can correctly classify unseen movie reviews into positive and negative sentiments. In the examples shown, all predictions match the actual labels, indicating that the model has effectively learned contextual patterns in the text data.

Although these examples show perfect predictions, the overall performance of the model is better represented by evaluation metrics such as accuracy and F1-score. The results confirm that the model generalizes well to new data.

######The model was evaluated using accuracy, precision, recall, and F1-score. The evaluation results show that the Transformer model performs well in classifying movie reviews as positive or negative.

######The accuracy score indicates the overall correctness of predictions, while precision and recall provide insights into classification performance for each class. The F1-score balances both precision and recall, giving a better measure of model performance.

######Sample predictions demonstrate that the model can correctly identify the sentiment of unseen reviews, indicating effective learning of contextual information.

##12. Analysis

######The Transformer model was successfully trained on the IMDB dataset for sentiment classification. The model achieved good accuracy and F1-score, indicating its ability to understand contextual meaning in text. Compared to traditional models, Transformers capture long-range dependencies effectively using attention mechanisms. However, they require more computational resources and training time.

##13. Model Comparison

######DistilBERT was used as a lightweight alternative to BERT. It provided faster training with comparable performance. BERT may achieve slightly higher accuracy due to its deeper architecture, but DistilBERT is more efficient for practical usage.

##14. Conclusion

######The Transformer-based model demonstrated strong performance in sentiment classification tasks. It effectively captured semantic meaning and context from text data. Although computationally expensive, Transformers outperform traditional machine learning models in NLP tasks.